In [1]:
import pandas as pd
import dask.dataframe as dd
import numpy as np
import datetime as dt
from datetime import date
import gc

In [2]:
def drop_duplicate_claims(df,filetype): 
    print('# of rows')
    print(df.shape[0])

    print()


    taf_subset = ['CLM_ID','MDCD_ALOWD_AMT','MDCD_PD_AMT','SRVC_BGN_DT','SRVC_END_DT']
 
    max_subset = ['SRVC_BGN_DT','SRVC_END_DT','MDCD_PYMT_AMT','CHRG_AMT']
    
    print(filetype)

    if filetype == 'taf': 

        # drop duplicated claims
        df = df.drop_duplicates(subset=taf_subset,keep='first')

        print('dropped duplicates based on '+str(taf_subset))


    else: 

        # drop duplicated claims
        df = df.drop_duplicates(subset=max_subset,keep='first')

        print('dropped duplicates based on '+str(max_subset))


    print('# of rows')
    print(df.shape[0])
    print()

    return df 

In [3]:
def drop_missing_zero_amounts(df): 
    
    print('dropping negative amounts')
    print('# of rows')
    print(df.shape[0])
    print()

    # drop rows with negative MDCD_PD_AMT and negative MDCD_ALOWD_AMT
    df = df.loc[(df['MDCD_PD_AMT']==0) & (df['MDCD_ALOWD_AMT']==0)]
    
    # drop rows with a 

    print('dropped rows with negative MDCD_PD_AMT or MDCD_ALOWD_AMT')
    print('# of rows')
    print(df.shape[0])
    print()

In [4]:
def winsorize_amounts(df, variable): 
    
    print('winsorizing amounts')
    
    #set upper limit and lower limit 
    upper_limit = df[variable].quantile(q=0.95)
    lower_limit = df[variable].quantile(q=0.05)

    print('lower limit')
    print(lower_limit)
    print()

    print('upper limit')
    print(upper_limit)
    print()
    print()

    # winsorize claims to lower limit (5th percentile)
    df[variable] = df[variable].mask((df[variable]<=lower_limit), lower_limit)
    df[variable] = df[variable].astype(float)

    # winsorize claims to upper limit (95th percentile)
    df[variable] = df[variable].mask((df[variable]>=upper_limit), upper_limit)
    df[variable] = df[variable].astype(float)

    return(df)

In [6]:
def estimate_allowed_amount(df, filetype): 
    if filetype == 'taf': 
        medicaid_paid_col = 'MDCD_PD_AMT'
        third_party_paid_col = 'TP_PD_AMT'
        bene_liability_col = 'BENE_LIABILITY_AMT'
        
    elif filetype == 'max': 
        medicaid_paid_col = 'MDCD_PYMT_AMT'
        third_party_paid_col = 'TP_PYMT_AMT'
        bene_liability_col = 'PATIENT_LIB_AMT'

    df['estimated_allowed_amount_A'] = df[medicaid_paid_col] + df[third_party_paid_col] + df[bene_liability_col]
    df['estimated_allowed_amount_B'] = df[medicaid_paid_col] 
    df['estimated_allowed_amount_C'] = df[medicaid_paid_col] + df[bene_liability_col]
    df['estimated_allowed_amount_D'] = df[medicaid_paid_col] + df[third_party_paid_col] 
    

    return df 

    

In [9]:

def area_wage_adjustment(df,year,col_to_adjust): 

    if year > 2014: 
            
        area_wages = pd.read_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/{year}_facility_area_wage_index.csv', 
                                 dtype={'BLG_PRVDR_NPI':'str', 'Wage_Index':'float'})
        area_wages = area_wages.rename(columns={'BLG_PRVDR_NPI':'NPI'})
        
    else: 
        
        area_wages = pd.read_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/python_code/medicaid_pmt_rates/wage_index/{year}_facility_area_wage_index.csv', 
                                 dtype={'NPI':'str', 'Wage_Index':'float'})

    area_wages['NPI'] = area_wages['NPI'].astype('str')


    df = df.merge(area_wages, how='left', on=['NPI'],indicator='npi_area_wage_merge')


    new_col_name = f'{col_to_adjust}_wage_adj'
    df[new_col_name] = (df[f'{col_to_adjust}']/df['Wage_Index'])
    df[new_col_name] = df[new_col_name].astype(float)
                  
    return df 

In [10]:
def calculate_descriptive_stats(df, state, variable): 
    
    print('calculating descriptive stats')
    
    stats = df[variable].describe().reset_index()
    stats['state'] = f'{state}'

    stats = stats.pivot(index='state',columns='index', values=variable).reset_index()
    stats['pmt_var'] = f'{variable}'
    stats['var'] = df[variable].var()


    return(stats)

In [11]:
# EXAMPLE: Calculate average state daily Medicaid reimbursement rates using TAF 2020-2021 files" 

states = ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 
'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY',
'LA', 'MA', 'MD', 'ME', 'MI', 'MO', 'MS', 'MT', 'NC',
'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 
'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 
'VT', 'WA', 'WI', 'WV', 'WY']

for year in [2020,2021]: 
    
    df_list = [] 
    
    for state in states: 
    
        # read in 2019 taf NF claims
        print('reading in data')
        claims = pd.read_parquet(f'/gpfs/data/cms-share/duas/56930/Nadia/data/medicaid_NF_pmt_rates/taf_lt_NF_claims/{year}/{state}',
                                engine='pyarrow')
        claims = claims.rename(columns={'BLG_PRVDR_NPI':'NPI'})
        claims['NPI'] = claims['NPI'].astype('str')


        print(claims.columns)
        
        print(f'{year} {state} read in')
        # print(test_taf_nf_claims.columns)

        claims = estimate_allowed_amount(claims, 'taf')
    
        claims['MDCD_PD_AMT'] = claims['MDCD_PD_AMT'].astype('float')
        claims['MDCD_ALOWD_AMT'] = claims['MDCD_ALOWD_AMT'].astype('float')
        claims['BENE_LIABILITY_AMT'] = claims['BENE_LIABILITY_AMT'].astype('float')
        claims['TP_PD_AMT'] = claims['TP_PD_AMT'].astype('float')
        claims['DAY_COUNT'] = claims['DAY_COUNT'].astype('float')

        

        claims = drop_duplicate_claims(claims,'taf')

        # claims = estimate_allowed_amount(claims, 'taf')
    
        claims = winsorize_amounts(claims, 'MDCD_ALOWD_AMT')
        claims['PER_DIEM'] = round((claims['MDCD_ALOWD_AMT']/claims['DAY_COUNT']),2)
    
    
        # calculate unadjusted paid per diem 
        unadjusted_state_rates = calculate_descriptive_stats(claims, state, 'PER_DIEM')
    
        
        ### AREA WAGE ADJUSTMENT 
        wage_adjusted_state_rates = area_wage_adjustment(claims, year, 'PER_DIEM')
        wage_adjusted_state_rates = calculate_descriptive_stats(wage_adjusted_state_rates, state, 'PER_DIEM_wage_adj')
    
     
        state_rates = pd.concat([unadjusted_state_rates, wage_adjusted_state_rates], axis=0)
            
        df_list.append(state_rates)
    
        # del acuity_adjusted_state_rates
        
    
        print(f'{year} {state} completed')
    
    state_rates = pd.concat(df_list, axis=0)
    
    print(f'{year} COMPLETED')
    state_rates.to_csv(f'/{directory_to_store_calculated_rates}/per_diems/taf_allowed_{year}_{date.today()}.csv')


reading in data
Index(['BENE_MSIS', 'SUBMTG_STATE_CD', 'CLM_ID', 'CLM_TYPE_CD',
       'CROSSOVER_CLM_IND', 'PYMT_LVL_IND', 'MDCD_ALOWD_AMT', 'MDCD_PD_AMT',
       'BENE_LIABILITY_AMT', 'TP_PD_AMT', 'MDCR_PD_AMT', 'SRVC_BGN_DT',
       'SRVC_END_DT', 'MDCD_ACMDTN_PD_AMT', 'MDCD_ANCLRY_PD_AMT', 'DAILY_RATE',
       'BLG_PRVDR_ID', 'NPI', 'BLG_PRVDR_TXNMY_CD', 'BLG_PRVDR_TYPE_CD',
       'SRVC_PRVDR_NPI', 'REV_CNTR_CD', 'TOS_CD', 'nf_claim', 'DAY_COUNT',
       'facility_npi'],
      dtype='object')
2020 AK read in
# of rows
12131

taf
dropped duplicates based on ['CLM_ID', 'MDCD_ALOWD_AMT', 'MDCD_PD_AMT', 'SRVC_BGN_DT', 'SRVC_END_DT']
# of rows
12131

winsorizing amounts
lower limit
2285.22

upper limit
37037.66


calculating descriptive stats
calculating descriptive stats
2020 AK completed
reading in data
Index(['BENE_MSIS', 'SUBMTG_STATE_CD', 'CLM_ID', 'CLM_TYPE_CD',
       'CROSSOVER_CLM_IND', 'PYMT_LVL_IND', 'MDCD_ALOWD_AMT', 'MDCD_PD_AMT',
       'BENE_LIABILITY_AMT', 'TP_PD_AMT', 

In [61]:
# EXAMPLE: Calculate average state daily Medicaid reimbursement rates using a subset of state MAX 2013-2024 files" 

sample_state_list = ['ID','LA',  'MI',  'MO',  'MS',  'NJ',  'PA',  'SD',  'UT',  'VT',  'WV', 'WY']

## RUN 2013 MAX rates 

for year in [2011, 2012, 2013, 2014]: 

    df_list = []
    
    for state in sample_state_list: 
            
 
    
        # read in max NF claims
        claims = pd.read_parquet(f'/gpfs/data/cms-share/duas/56930/Nadia/data/medicaid_NF_pmt_rates/max_lt_NF_claims/{year}/{state}', 
                                 engine='pyarrow')
        claims['NPI'] = claims['NPI'].astype(str)
    
        print(f'{year} {state} read in')

        print(claims.columns)
        # print(test_taf_nf_claims.columns)
    
        claims['CHRG_AMT'] = claims['CHRG_AMT'].astype('float')
        claims['DAY_COUNT'] = claims['DAY_COUNT'].astype('float')
    
    
        claims = drop_duplicate_claims(claims,'max')
    
        claims = winsorize_amounts(claims, 'CHRG_AMT')
    
        # calculate paid paid per diem 
        claims['PER_DIEM'] = round((claims['CHRG_AMT']/claims['DAY_COUNT']),2)
        
    
        unadjusted_state_rates = calculate_descriptive_stats(claims, state, 'PER_DIEM')

        ### AREA WAGE ADJUSTMENT 


        
        wage_adjusted_state_rates = area_wage_adjustment(claims, year, 'PER_DIEM')
        wage_adjusted_state_rates = calculate_descriptive_stats(wage_adjusted_state_rates, state, 'PER_DIEM_wage_adj')
    
        state_rates = pd.concat([unadjusted_state_rates, wage_adjusted_state_rates], axis=0)
        df_list.append(state_rates)

    
        print(f'{year} {state} completed')
    
    state_rates = pd.concat(df_list, axis=0)
    print(state_rates.head(10))

    state_rates.to_csv(f'/gpfs/data/cms-share/duas/56930/Nadia/data/medicaid_NF_pmt_rates/calculated_rates/per_diems/sample_states/max_{year}_{date.today()}.csv')

2011 ID read in
Index(['BENE_MSIS', 'BENE_ID', 'MSIS_ID', 'PRVDR_ID_NMBR', 'NPI',
       'MSNG_ELG_DATA', 'MSIS_TOS', 'MAX_TOS', 'TYPE_CLM_CD', 'ADJUST_CD',
       'MDCD_PYMT_AMT', 'CHRG_AMT', 'MDCR_COINSUR_PYMT_AMT',
       'MDCR_DED_PYMT_AMT', 'TP_PYMT_AMT', 'PATIENT_LIB_AMT',
       'NRSNG_FAC_DAY_CNT', 'EL_MDCR_XOVR_CLM_BSD_CD', 'SRVC_BGN_DT',
       'SRVC_END_DT', 'ALOWD_AMT', 'DAY_COUNT'],
      dtype='object')
# of rows
22378

max
dropped duplicates based on ['SRVC_BGN_DT', 'SRVC_END_DT', 'MDCD_PYMT_AMT', 'CHRG_AMT']
# of rows
14074

winsorizing amounts
lower limit
1014.0

upper limit
8264.0


calculating descriptive stats
calculating descriptive stats
2011 ID completed
2011 LA read in
Index(['BENE_MSIS', 'BENE_ID', 'MSIS_ID', 'PRVDR_ID_NMBR', 'NPI',
       'MSNG_ELG_DATA', 'MSIS_TOS', 'MAX_TOS', 'TYPE_CLM_CD', 'ADJUST_CD',
       'MDCD_PYMT_AMT', 'CHRG_AMT', 'MDCR_COINSUR_PYMT_AMT',
       'MDCR_DED_PYMT_AMT', 'TP_PYMT_AMT', 'PATIENT_LIB_AMT',
       'NRSNG_FAC_DAY_CNT', 'EL_MD